In [18]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import requests

# =====================================================================
# ⚙️ HARDCODED USER STRATEGY CONFIGURATIONS (Modify values here)
# =====================================================================
CONFIG_ETF_TICKER   = "SPY"             # Target ETF Basket Universe
CONFIG_NUM_STOCKS   = 5                 # Select Top N Stocks to Own
CONFIG_BACKTEST_YRS = "5y"              # "1y", "2y", "5y", "10y", "max"
CONFIG_ALLOCATIONS  = [20, 20, 20, 20, 20] # Custom Allocation Weights

# =====================================================================

class IndexPulseEngine:
    def __init__(self):
        self.df_constituents = None   
        self.df_rankings = None       
        self.df_live_signals = None   
        self.df_metrics = None        
        self.df_stress_tests = None   # Audit DataFrame #5: Window Comparisons
        self.returns_matrix = None    

    def extract_universe(self, etf_ticker: str) -> list:
        target = str(etf_ticker).upper().strip()
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        try:
            if "SPY" in target:
                url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
                tables = pd.read_html(requests.get(url, headers=headers).text)
                return [str(s).strip().replace('.', '-') for s in tables[0]['Symbol'].dropna().tolist()]
            elif "QQQ" in target:
                url = "https://en.wikipedia.org/wiki/Nasdaq-100"
                tables = pd.read_html(requests.get(url, headers=headers).text)
                component_table = next(t for t in tables if 'Ticker' in t.columns or 'Symbol' in t.columns)
                col = 'Ticker' if 'Ticker' in component_table.columns else 'Symbol'
                return [str(s).strip().replace('.', '-') for s in component_table[col].dropna().tolist()]
        except Exception:
            pass

        try:
            url = f"https://financialmodelingprep.com/api/v3/etf-holder/{target}?apikey=demo"
            response = requests.get(url, headers=headers, timeout=5)
            if response.status_code == 200:
                json_data = response.json()
                if isinstance(json_data, list) and len(json_data) > 0:
                    return [str(item['asset']).strip().replace('.', '-') for item in json_data if 'asset' in item]
        except Exception:
            pass

        fallback_map = {
            "DIA": ["UNH", "GS", "MSFT", "HD", "CAT", "CRM", "AMGN", "V", "BA", "HON", "JNJ", "AXP", "PG", "AAPL", "IBM", "MCD", "MMM", "DIS", "KO", "CSCO", "CVX", "NKE", "WMT", "MRK", "VZ", "TRV", "INTC", "DOW", "SHW"],
            "PHO": ["AWK", "XYL", "AOS", "WTS", "ECL", "ROP", "FI", "TMO", "DHR", "PNR", "ITW", "CGNX", "CWT", "MWA", "SBS", "ETN", "FELE", "AWR", "WTRG", "MGEE", "SJW", "YORW", "CTOS", "XLY"],
            "XLK": ["MSFT", "AAPL", "NVDA", "AVGO", "AMD", "QCOM", "CRM", "INTC", "AMAT", "NOW", "ADI", "MU", "LRCX", "KLAC", "PANW", "FTNT", "SNPS", "CDNS", "APH", "MSI", "ORCL", "IBM"]
        }
        return fallback_map.get(target, [])

    def _calculate_performance_metrics(self):
        """
        Computes standard risk metrics along with win/loss distributions.
        """
        initial_val = 10000.0
        metrics_summary = []

        for column in ['Baseline_BH', 'Strategy_Momentum']:
            equity_series = self.returns_matrix[column]
            daily_returns = equity_series.pct_change().dropna()
            
            final_value = equity_series.iloc[-1]
            total_return_pct = ((final_value - initial_val) / initial_val) * 100
            
            years_count = max((equity_series.index[-1] - equity_series.index[0]).days / 365.25, 1 / 252)
            cagr_pct = (((final_value / initial_val) ** (1 / years_count)) - 1) * 100
            annualized_vol_pct = (daily_returns.std() * np.sqrt(252)) * 100
            sharpe_ratio = (cagr_pct / annualized_vol_pct) if annualized_vol_pct > 0 else 0.0
            
            running_peaks = equity_series.cummax()
            max_drawdown_pct = ((equity_series - running_peaks) / running_peaks).min() * 100
            
            # Daily Distribution Elements
            pos_days = daily_returns[daily_returns > 0]
            neg_days = daily_returns[daily_returns < 0]
            
            metrics_summary.append({
                "Strategy Track": "Index Buy & Hold" if column == "Baseline_BH" else "Matrix Momentum",
                "Total Value ($)": f"${final_value:,.2f}",
                "Total Return": f"{total_return_pct:.2f}%",
                "Annualized Return (CAGR)": f"{cagr_pct:.2f}%",
                "Average Volatility": f"{annualized_vol_pct:.2f}%",
                "Sharpe Ratio": f"{sharpe_ratio:.2f}",
                "Max Drawdown": f"{max_drawdown_pct:.2f}%",
                "Total Positive Days": int(len(pos_days)),
                "Total Negative Days": int(len(neg_days)),
                "Avg Gain on Positive Days": f"{(pos_days.mean() * 100):.2f}%" if len(pos_days) > 0 else "0.00%",
                "Avg Loss on Negative Days": f"{(neg_days.mean() * 100):.2f}%" if len(neg_days) > 0 else "0.00%"
            })
            
        self.df_metrics = pd.DataFrame(metrics_summary).set_index("Strategy Track").T

    def _execute_window_stress_tests(self):
        """
        Isolates the top 5 worst and best rolling 5-day windows for both tracks
        and explicitly benchmarks the opposing asset's behavior during those same windows.
        """
        stress_records = []
        
        # Calculate a rolling 5-day (weekly) return matrix across the equity timeline
        weekly_returns = self.returns_matrix.pct_change(periods=5).dropna()
        
        for baseline, label in [('Baseline_BH', 'Index B&H'), ('Strategy_Momentum', 'Momentum Strat')]:
            # Locate the absolute top 5 worst periods
            worst_dates = weekly_returns[baseline].nsmallest(5).index
            for idx, end_date in enumerate(worst_dates):
                start_date = weekly_returns.index[weekly_returns.index.get_loc(end_date) - 5]
                b_ret = ((self.returns_matrix['Baseline_BH'].loc[end_date] / self.returns_matrix['Baseline_BH'].loc[start_date]) - 1) * 100
                m_ret = ((self.returns_matrix['Strategy_Momentum'].loc[end_date] / self.returns_matrix['Strategy_Momentum'].loc[start_date]) - 1) * 100
                stress_records.append({
                    "Condition Category": f"Top 5 Worst Weeks for {label}",
                    "Rank": f"Worst #{idx + 1}",
                    "Window Frame": f"{start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}",
                    "Index B&H Return": f"{b_ret:.2f}%",
                    "Momentum Strat Return": f"{m_ret:.2f}%"
                })

            # Locate the absolute top 5 best periods
            best_dates = weekly_returns[baseline].nlargest(5).index
            for idx, end_date in enumerate(best_dates):
                start_date = weekly_returns.index[weekly_returns.index.get_loc(end_date) - 5]
                b_ret = ((self.returns_matrix['Baseline_BH'].loc[end_date] / self.returns_matrix['Baseline_BH'].loc[start_date]) - 1) * 100
                m_ret = ((self.returns_matrix['Strategy_Momentum'].loc[end_date] / self.returns_matrix['Strategy_Momentum'].loc[start_date]) - 1) * 100
                stress_records.append({
                    "Condition Category": f"Top 5 Best Weeks for {label}",
                    "Rank": f"Best #{idx + 1}",
                    "Window Frame": f"{start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}",
                    "Index B&H Return": f"{b_ret:.2f}%",
                    "Momentum Strat Return": f"{m_ret:.2f}%"
                })
                
        self.df_stress_tests = pd.DataFrame(stress_records).set_index(["Condition Category", "Rank"])

    def run_analysis(self, etf_ticker: str, num_stocks: int, horizon: str, allocations: list):
        target_etf = str(etf_ticker).upper().strip()
        total_weight_sum = sum(allocations) if sum(allocations) > 0 else 100
        normalized_weights = [w / total_weight_sum for w in allocations]
        
        print(f"📡 Step 1: Mapping constituents universe for {target_etf}...")
        raw_components = self.extract_universe(target_etf)
        if not raw_components:
            raise ValueError(f"❌ Universe metadata for '{target_etf}' is unretrievable.")
            
        sanitized = sorted(list(set([str(t).strip().upper().replace('.', '-') for t in raw_components])))
        all_tickers = list(set(sanitized + [target_etf]))
        
        print(f"📥 Step 2: Parallel downloading data metrics across {len(sanitized)} underlying assets...")
        data = yf.download(all_tickers, period=horizon, interval="1d", auto_adjust=False, progress=False, threads=True)
        if data.empty or 'Close' not in data:
            raise RuntimeError("❌ Historical pricing data stream timeout.")
            
        close_matrix = data['Close'].ffill().fillna(0.0).round(2)
        open_matrix = data['Open'].ffill().fillna(0.0).round(2)
        valid_assets = [c for c in sanitized if c in close_matrix.columns and c != target_etf]
        
        self.df_constituents = pd.DataFrame({"Ticker Symbol": valid_assets}, index=range(1, len(valid_assets) + 1))
        
        print("⚡ Step 3: Generating vectorized momentum matrices and audits...")
        returns_df = close_matrix[valid_assets].pct_change()
        rank_history = {}
        lookback = 63
        
        for i in range(lookback, len(close_matrix), 21):
            dt = close_matrix.index[i]
            total_returns = (close_matrix[valid_assets].iloc[i] / close_matrix[valid_assets].iloc[i - lookback]) - 1
            volatilities = returns_df.iloc[i - lookback:i].std() * np.sqrt(252)
            scores = (total_returns / (volatilities + 1e-8)).dropna()
            
            sorted_assets = sorted(scores.items(), key=lambda x: (-round(x[1], 4), x[0]))
            rank_history[dt] = [asset[0] for asset in sorted_assets]

        ranking_dates = sorted(list(rank_history.keys()))
        audit_rows = []
        for dt in ranking_dates:
            row = {"Rebalance Date": dt.strftime('%Y-%m-%d')}
            for r in range(num_stocks):
                row[f"Rank #{r+1}"] = rank_history[dt][r] if r < len(rank_history[dt]) else "N/A"
            audit_rows.append(row)
        self.df_rankings = pd.DataFrame(audit_rows).set_index("Rebalance Date")
        
        p_start_live = close_matrix[valid_assets].iloc[-lookback]
        p_end_live = close_matrix[valid_assets].iloc[-1]
        returns_live = (p_end_live / p_start_live) - 1
        vols_live = returns_df.iloc[-lookback:].std() * np.sqrt(252)
        live_scores = (returns_live / (vols_live + 1e-8)).dropna()
        live_ranked = [a[0] for a in sorted(live_scores.items(), key=lambda x: (-round(x[1], 4), x[0]))]
        
        live_rows = []
        for r in range(num_stocks):
            tk = live_ranked[r]
            live_rows.append({
                "Live Rank": f"Rank #{r + 1}",
                "Stock Ticker": tk,
                "Momentum Score": round(live_scores[tk], 4),
                "Target Allocation Weight": f"{(normalized_weights[r] * 100):.1f}%"
            })
        self.df_live_signals = pd.DataFrame(live_rows).set_index("Live Rank")

        # Core Walk-Forward Simulation Vector Loop
        cash = initial_capital = 10000
        shares = {c: 0.0 for c in valid_assets}
        equity_curve, timeline = [], []
        last_month = -1
        
        for date in close_matrix.index[close_matrix.index >= ranking_dates[0]]:
            if date.month != last_month and date in rank_history:
                last_month = date.month
                top_perf = rank_history[date][:num_stocks]
                opens = open_matrix.loc[date]
                
                nav = cash + sum(u * opens[a] for a, u in shares.items() if u > 0 and a in opens)
                cash = nav
                shares = {c: 0.0 for c in valid_assets}
                
                for idx, tk in enumerate(top_perf):
                    if idx < len(normalized_weights) and tk in opens and opens[tk] > 0:
                        allocated = nav * normalized_weights[idx]
                        if cash >= allocated:
                            cash -= allocated
                            shares[tk] = (allocated * 0.9995) / opens[tk]
                            
            evening_nav = cash + sum(u * close_matrix.loc[date, a] for a, u in shares.items() if u > 0)
            equity_curve.append(evening_nav)
            timeline.append(date)
            
        etf_close = close_matrix.loc[timeline, target_etf]
        etf_bh = initial_capital * (etf_close / open_matrix.loc[timeline[0], target_etf]).values
        self.returns_matrix = pd.DataFrame({"Baseline_BH": etf_bh, "Strategy_Momentum": equity_curve}, index=timeline)
        
        # Populate metrics scorecard and cross-sectional window stress logs
        self._calculate_performance_metrics()
        self._execute_window_stress_tests()
        print(f"🎯 Complete! Comprehensive research analytics database generated for {target_etf}.")

    def display_performance_chart(self, etf_ticker: str):
        if self.returns_matrix is None:
            print("⚠️ Warning: Data missing. Call .run_analysis() first.")
            return
        target_label = str(etf_ticker).upper().strip()
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=self.returns_matrix.index, y=self.returns_matrix['Baseline_BH'], name=f'{target_label} Baseline Buy & Hold', line=dict(color='gray', width=1.5)))
        fig.add_trace(go.Scatter(x=self.returns_matrix.index, y=self.returns_matrix['Strategy_Momentum'], name='Custom Matrix Momentum Strategy', line=dict(color='blue', width=2.5)))
        fig.update_layout(template="plotly_white", title=f"📊 Performance Evaluation: Full {target_label} Asset Matrix Universe", xaxis_title="Date Range Timeline", yaxis_title="Account Balance ($)", height=500, legend=dict(orientation="h", y=1.08), hovermode="x unified")
        fig.show()

In [19]:
engine = IndexPulseEngine()

engine.run_analysis(
    etf_ticker=CONFIG_ETF_TICKER, 
    num_stocks=CONFIG_NUM_STOCKS, 
    horizon=CONFIG_BACKTEST_YRS, 
    allocations=CONFIG_ALLOCATIONS
)

📡 Step 1: Mapping constituents universe for SPY...


C:\Users\reywa\AppData\Local\Temp\ipykernel_13612\1005763701.py:32: FutureWarning:

Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.



📥 Step 2: Parallel downloading data metrics across 503 underlying assets...
⚡ Step 3: Generating vectorized momentum matrices and audits...
🎯 Complete! Comprehensive research analytics database generated for SPY.


In [23]:
# Ensure all stocks were pulled
len(engine.df_constituents)

503

In [24]:
# View the last 10 historical rebalance cycles leading up to this month
engine.df_rankings.tail(10)

,Rank #1,Rank #2,Rank #3,Rank #4,Rank #5
Rebalance Date,,,,,
2025-09-03,WDC,LITE,GOOG,GOOGL,ECHO
2025-10-02,WDC,SNDK,APP,LITE,STX
2025-10-31,SNDK,MU,WDC,CIEN,INTC
2025-12-02,SNDK,CIEN,MU,AMAT,WBD
2026-01-02,LITE,LLY,WBD,JKHY,DLTR
2026-02-03,SNDK,FDX,PNC,FCX,LUV
2026-03-05,FDX,LMT,TPL,SNDK,MRNA
2026-04-06,APA,LYB,SNDK,CASY,DOW
2026-05-05,MRVL,INTC,CASY,ON,LITE


In [20]:
# View the entire cross-sectional window test database
pd.set_option('display.max_rows', 50)
engine.df_stress_tests

Window Frame  \
Condition Category                   Rank                                 
Top 5 Worst Weeks for Index B&H      Worst #1  2025-04-01 to 2025-04-08   
                                     Worst #2  2022-06-07 to 2022-06-14   
                                     Worst #3  2025-03-31 to 2025-04-07   
                                     Worst #4  2025-03-28 to 2025-04-04   
                                     Worst #5  2022-06-06 to 2022-06-13   
Top 5 Best Weeks for Index B&H       Best #1   2025-04-08 to 2025-04-15   
                                     Best #2   2025-04-21 to 2025-04-28   
                                     Best #3   2025-04-07 to 2025-04-14   
                                     Best #4   2022-05-20 to 2022-05-27   
                                     Best #5   2022-03-14 to 2022-03-21   
Top 5 Worst Weeks for Momentum Strat Worst #1  2022-06-10 to 2022-06-17   
                                     Worst #2  2026-06-03 to 2026-06-10   
                                     Worst #3  2023-08-02 to 2023-08-09   
                                     Worst #4  2022-06-07 to 2022-06-14   
                                     Worst #5  2023-08-03 to 2023-08-10   
Top 5 Best Weeks for Momentum Strat  Best #1   2024-11-04 to 2024-11-11   
                                     Best #2   2025-10-22 to 2025-10-29   
                                     Best #3   2024-11-01 to 2024-11-08   
                                     Best #4   2025-11-04 to 2025-11-11   
                                     Best #5   2025-11-20 to 2025-11-28   

                                              Index B&H Return  \
Condition Category                   Rank                        
Top 5 Worst Weeks for Index B&H      Worst #1          -11.50%   
                                     Worst #2          -10.07%   
                                     Worst #3           -9.83%   
                                     Worst #4           -9.07%   
                                     Worst #5           -8.93%   
Top 5 Best Weeks for Index B&H       Best #1             8.28%   
                                     Best #2             7.19%   
                                     Best #3             6.89%   
                                     Best #4             6.58%   
                                     Best #5             6.57%   
Top 5 Worst Weeks for Momentum Strat Worst #1           -6.14%   
                                     Worst #2           -3.82%   
                                     Worst #3           -0.97%   
                                     Worst #4          -10.07%   
                                     Worst #5           -0.65%   
Top 5 Best Weeks for Momentum Strat  Best #1             5.08%   
                                     Best #2             2.93%   
                                     Best #3             4.75%   
                                     Best #4             1.15%   
                                     Best #5             4.73%   

                                              Momentum Strat Return  
Condition Category                   Rank                            
Top 5 Worst Weeks for Index B&H      Worst #1                -4.78%  
                                     Worst #2               -12.64%  
                                     Worst #3                -6.27%  
                                     Worst #4                -3.14%  
                                     Worst #5                -9.95%  
Top 5 Best Weeks for Index B&H       Best #1                  5.11%  
                                     Best #2                  4.16%  
                                     Best #3                  6.17%  
                                     Best #4                 11.80%  
                                     Best #5                  3.05%  
Top 5 Worst Weeks for Momentum Strat Worst #1               -13.03%  
                                     Worst #2

In [21]:
# Prints basic risk metrics along with positive/negative daily distributions
engine.df_metrics

Strategy Track,Index Buy & Hold,Matrix Momentum
Total Value ($),"$16,582.73","$44,822.61"
Total Return,65.83%,348.23%
Annualized Return (CAGR),11.25%,37.19%
Average Volatility,17.49%,28.55%
Sharpe Ratio,0.64,1.30
Max Drawdown,-25.36%,-21.16%
Total Positive Days,644,668
Total Negative Days,544,522
Avg Gain on Positive Days,0.77%,1.23%
Avg Loss on Negative Days,-0.80%,-1.25%


In [25]:
# Plot the historical strategy performance against the index buy-and-hold baseline
engine.display_performance_chart(etf_ticker=CONFIG_ETF_TICKER)

In [22]:
# Print the current month's active trading matrix
engine.df_live_signals

,Stock Ticker,Momentum Score,Target Allocation Weight
Live Rank,,,
Rank #1,HUM,2.6985,20.0%
Rank #2,STX,2.6976,20.0%
Rank #3,SNDK,2.6183,20.0%
Rank #4,MU,2.3529,20.0%
Rank #5,INTC,2.1736,20.0%


In [28]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests

# =====================================================================
# ⚙️ HARDCODED PARAMETRIC OPTIMIZATION CONFIGURATIONS
# =====================================================================
SEARCH_TARGET_ETF = "SPY"
SEARCH_HORIZON    = "5y"

# 🎯 SCORECARD OBJECTIVE IMPORTANCE WEIGHTS (Must add up to 100%)
WEIGHT_TOTAL_RETURN = 25   # % weight assigned to maximizing return
WEIGHT_MAX_DRAWDOWN = 35   # % weight assigned to minimizing downside risk
WEIGHT_SHARPE_RATIO = 40   # % weight assigned to risk-adjusted consistency

GRID_PARAM_SPACE = {
    3: [
        [33.3, 33.3, 33.3],       
        [50.0, 35.0, 15.0],       
        [45.0, 35.0, 20.0]        
    ],
    5: [
        [20.0, 20.0, 20.0, 20.0, 20.0], 
        [35.0, 25.0, 20.0, 15.0, 5.0],  
        [30.0, 25.0, 20.0, 15.0, 10.0]  
    ],
    7: [
        [14.3, 14.3, 14.3, 14.3, 14.3, 14.3, 14.3], 
        [30.0, 22.0, 18.0, 12.0, 10.0, 5.0, 3.0],   
        [25.0, 20.0, 18.0, 15.0, 10.0, 8.0, 4.0]    
    ],
    10: [
        [10.0]*10,                                              
        [25.0, 18.0, 15.0, 12.0, 10.0, 8.0, 5.0, 4.0, 2.0, 1.0], 
        [20.0, 16.0, 14.0, 12.0, 10.0, 9.0, 8.0, 6.0, 3.0, 2.0]  
    ]
}

# =====================================================================

class QuantGridSearchOptimizer:
    def __init__(self, etf_ticker: str, horizon: str):
        self.etf_target = etf_ticker.upper().strip()
        self.horizon = horizon
        self.close_matrix = None
        self.open_matrix = None
        self.returns_df = None
        self.valid_assets = []
        
        # Output Scorecard Storage Repositories
        self.df_optimization_results = None
        self.df_best_return = None
        self.df_best_drawdown = None
        self.df_best_sharpe = None

    def _fetch_all_underlying_data(self):
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        try:
            url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies" if "SPY" in self.etf_target else "https://en.wikipedia.org/wiki/Nasdaq-100"
            tables = pd.read_html(requests.get(url, headers=headers).text)
            component_table = tables[0]
            col = 'Symbol' if 'Symbol' in component_table.columns else 'Ticker'
            raw_tickers = [str(s).strip().replace('.', '-') for s in component_table[col].dropna().tolist()]
        except Exception:
            raw_tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "BRK-B", "LLY", "AVGO", "JPM"] 
            
        sanitized = sorted(list(set([t.upper() for t in raw_tickers if len(t) <= 5])))
        all_tickers = list(set(sanitized + [self.etf_target]))
        
        data = yf.download(all_tickers, period=self.horizon, interval="1d", auto_adjust=False, progress=False, threads=True)
        self.close_matrix = data['Close'].ffill().fillna(0.0).round(2)
        self.open_matrix = data['Open'].ffill().fillna(0.0).round(2)
        self.valid_assets = [c for c in sanitized if c in self.close_matrix.columns and c != self.etf_target]
        self.returns_df = self.close_matrix[self.valid_assets].pct_change()

    def execute_weighted_sweep(self, param_space: dict, w_return: float, w_drawdown: float, w_sharpe: float):
        """ Runs a grid sweep weighted by customized objective metrics. """
        if round(w_return + w_drawdown + w_sharpe, 2) != 100.0:
            raise ValueError(f"❌ Error: Objective weights must equal 100%. Current sum: {w_return + w_drawdown + w_sharpe}%")
            
        self._fetch_all_underlying_data()
        
        # 🎯 THE AUDIT LEDGER CLAUSE
        print(f"📊 [AUDIT]: Verified {len(self.valid_assets)} distinct asset components mapped and analyzed from the {self.etf_target} tracking ledger.")
        
        lookback = 63
        rank_history = {}
        
        for i in range(lookback, len(self.close_matrix), 21):
            dt = self.close_matrix.index[i]
            total_returns = (self.close_matrix[self.valid_assets].iloc[i] / self.close_matrix[self.valid_assets].iloc[i - lookback]) - 1
            volatilities = self.returns_df.iloc[i - lookback:i].std() * np.sqrt(252)
            scores = (total_returns / (volatilities + 1e-8)).dropna()
            rank_history[dt] = [asset[0] for asset in sorted(scores.items(), key=lambda x: (-round(x[1], 4), x[0]))]

        ranking_dates = sorted(list(rank_history.keys()))
        sweep_records = []
        
        for n_val, weights_permutations in param_space.items():
            for weights in weights_permutations:
                w_sum = sum(weights)
                norm_weights = [w / w_sum for w in weights]
                
                cash = initial_capital = 10000.0
                shares = {c: 0.0 for c in self.valid_assets}
                equity_curve = []
                last_month = -1
                
                for date in self.close_matrix.index[self.close_matrix.index >= ranking_dates[0]]:
                    if date.month != last_month and date in rank_history:
                        last_month = date.month
                        top_perf = rank_history[date][:n_val]
                        opens = self.open_matrix.loc[date]
                        
                        nav = cash + sum(u * opens[a] for a, u in shares.items() if u > 0 and a in opens)
                        cash = nav
                        shares = {c: 0.0 for c in self.valid_assets}
                        
                        for idx, tk in enumerate(top_perf):
                            if idx < len(norm_weights) and tk in opens and opens[tk] > 0:
                                allocated = nav * norm_weights[idx]
                                if cash >= allocated:
                                    cash -= allocated
                                    shares[tk] = (allocated * 0.9995) / opens[tk]
                                    
                    evening_nav = cash + sum(u * self.close_matrix.loc[date, a] for a, u in shares.items() if u > 0)
                    equity_curve.append(evening_nav)
                
                equity_series = pd.Series(equity_curve)
                daily_ret = equity_series.pct_change().dropna()
                
                total_return_pct = ((equity_series.iloc[-1] - initial_capital) / initial_capital) * 100
                years = len(equity_series) / 252
                cagr_pct = (((equity_series.iloc[-1] / initial_capital) ** (1 / years)) - 1) * 100
                vol_pct = (daily_ret.std() * np.sqrt(252)) * 100
                sharpe = (cagr_pct / vol_pct) if vol_pct > 0 else 0.0
                
                running_peaks = equity_series.cummax()
                max_dd_pct = ((equity_series - running_peaks) / running_peaks).min() * 100
                
                sweep_records.append({
                    "Top N choice": n_val,
                    "Allocation Config": str([round(x, 1) for x in weights]),
                    "Total Return (%)": total_return_pct,
                    "Max Drawdown (%)": max_dd_pct,
                    "Sharpe Ratio": sharpe
                })
                
        df = pd.DataFrame(sweep_records)
        
        # Generate operational data sorting filters
        df['Rank_Return'] = df['Total Return (%)'].rank(ascending=False)
        df['Rank_Sharpe'] = df['Sharpe Ratio'].rank(ascending=False)
        df['Rank_MaxDD']  = df['Max Drawdown (%)'].rank(ascending=True)
        
        # ⚡ APPLY CUSTOM USER IMPORTANCE COEFFICIENTS
        df['Composite Score'] = (
            (df['Rank_Return'] * (w_return / 100.0)) + 
            (df['Rank_Sharpe'] * (w_sharpe / 100.0)) + 
            (df['Rank_MaxDD']  * (w_drawdown / 100.0))
        )
        
        # Sort master optimization table
        self.df_optimization_results = df.sort_values(by="Composite Score").drop(columns=['Rank_Return', 'Rank_Sharpe', 'Rank_MaxDD']).reset_index(drop=True)
        self.df_optimization_results.index += 1
        
        # 🔒 ISOLATE DEDICATED CATEGORY ISOLATION CHANNELS
        self.df_best_return   = df.sort_values(by="Total Return (%)", ascending=False).iloc[[0]].drop(columns=['Rank_Return', 'Rank_Sharpe', 'Rank_MaxDD', 'Composite Score']).reset_index(drop=True)
        self.df_best_drawdown = df.sort_values(by="Max Drawdown (%)", ascending=False).iloc[[0]].drop(columns=['Rank_Return', 'Rank_Sharpe', 'Rank_MaxDD', 'Composite Score']).reset_index(drop=True)
        self.df_best_sharpe   = df.sort_values(by="Sharpe Ratio", ascending=False).iloc[[0]].drop(columns=['Rank_Return', 'Rank_Sharpe', 'Rank_MaxDD', 'Composite Score']).reset_index(drop=True)
        
        print("\n" + "="*75)
        print("🏆 WEIGHTED OPTIMIZATION LEADERBOARD COMPILED")
        print("="*75)
        best = self.df_optimization_results.iloc[0]
        print(f"🥇 Weighted Winner: N={best['Top N choice']} | Weights={best['Allocation Config']}")
        print(f"📈 Return: {best['Total Return (%)']:.2f}% | Max DD: {best['Max Drawdown (%)']:.2f}% | Sharpe: {best['Sharpe Ratio']:.2f}")
        print("="*75 + "\n")

In [29]:
optimizer = QuantGridSearchOptimizer(etf_ticker=SEARCH_TARGET_ETF, horizon=SEARCH_HORIZON)

# Pass global configurations along with your exact percentage priority preferences
optimizer.execute_weighted_sweep(
    param_space=GRID_PARAM_SPACE,
    w_return=WEIGHT_TOTAL_RETURN,
    w_drawdown=WEIGHT_MAX_DRAWDOWN,
    w_sharpe=WEIGHT_SHARPE_RATIO
)

C:\Users\reywa\AppData\Local\Temp\ipykernel_13612\3089520466.py:61: FutureWarning:

Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.



📊 [AUDIT]: Verified 503 distinct asset components mapped and analyzed from the SPY tracking ledger.

🏆 WEIGHTED OPTIMIZATION LEADERBOARD COMPILED
🥇 Weighted Winner: N=10 | Weights=[25.0, 18.0, 15.0, 12.0, 10.0, 8.0, 5.0, 4.0, 2.0, 1.0]
📈 Return: 575.47% | Max DD: -24.08% | Sharpe: 1.66



In [30]:
print("📈 Absolute Best Total Return Configuration:")
display(optimizer.df_best_return)

print("\n🛡️ Absolute Best Max Drawdown Shield Configuration:")
display(optimizer.df_best_drawdown)

print("\n📊 Absolute Best Sharpe Risk-Adjusted Profile Configuration:")
display(optimizer.df_best_sharpe)

📈 Absolute Best Total Return Configuration:


,Top N choice,Allocation Config,Total Return (%),Max Drawdown (%),Sharpe Ratio
0,3,"[50.0, 35.0, 15.0]",729.181988,-29.737589,1.444188



🛡️ Absolute Best Max Drawdown Shield Configuration:


,Top N choice,Allocation Config,Total Return (%),Max Drawdown (%),Sharpe Ratio
0,5,"[20.0, 20.0, 20.0, 20.0, 20.0]",348.22614,-21.162735,1.307259



📊 Absolute Best Sharpe Risk-Adjusted Profile Configuration:


,Top N choice,Allocation Config,Total Return (%),Max Drawdown (%),Sharpe Ratio
0,10,"[10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10....",457.958418,-22.535347,1.713197


In [31]:
# Displays every permutation sorted by your custom composite scoring weights
optimizer.df_optimization_results

,Top N choice,Allocation Config,Total Return (%),Max Drawdown (%),Sharpe Ratio,Composite Score
1,10,"[25.0, 18.0, 15.0, 12.0, 10.0, 8.0, 5.0, 4.0, ...",575.467176,-24.080632,1.659137,4.65
2,10,"[20.0, 16.0, 14.0, 12.0, 10.0, 9.0, 8.0, 6.0, ...",552.843682,-24.016008,1.699658,4.85
3,3,"[50.0, 35.0, 15.0]",729.181988,-29.737589,1.444188,4.90
4,3,"[45.0, 35.0, 20.0]",582.388350,-32.628344,1.366851,5.10
5,7,"[25.0, 20.0, 18.0, 15.0, 10.0, 8.0, 4.0]",542.179151,-24.445138,1.583012,5.20
6,5,"[35.0, 25.0, 20.0, 15.0, 5.0]",595.062565,-23.476653,1.521048,5.65
7,7,"[30.0, 22.0, 18.0, 12.0, 10.0, 5.0, 3.0]",531.898243,-24.555844,1.500198,5.90
8,10,"[10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10....",457.958418,-22.535347,1.713197,6.75
9,5,"[30.0, 25.0, 20.0, 15.0, 10.0]",517.054591,-25.175490,1.450873,6.85
10,3,"[33.3, 33.3, 33.3]",526.296538,-32.243820,1.313758,7.10
